## PRÁCTICA 2: Aprendizaje Semi-supervisado y en Una Clase

### Cargar dataset

##### Trabajaremos con el conjunto de datos CIFAR-100 (50.000 instancias para entrenamiento y 10.000 para test), en el cual eliminaremos el 80% de sus etiquetas y realizaremos las siguientes particiones:

#####    40.000 instancias de entrenamiento no etiquetadas
#####    10.000 instancias de entrenamiento etiquetadas
#####    10.000 instancias de test etiquetadas

In [16]:
import tensorflow as tf
import numpy as np

In [17]:
labeled_data = 0.2 # Vamos a usar el etiquetado de sólo el 20% de los datos
np.random.seed(42)

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar100.load_data()

indexes = np.arange(len(x_train))
np.random.shuffle(indexes)
ntrain_data = int(labeled_data*len(x_train))
unlabeled_train = x_train[indexes[ntrain_data:]]
x_train = x_train[indexes[:ntrain_data]]
y_train = y_train[indexes[:ntrain_data]]

In [20]:
# TODO: Haz el preprocesado que necesites aquí (si lo necesitas)

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

x_train = x_train.reshape(-1, 32, 32, 3).astype("float32") / 255.0
x_test = x_test.reshape(-1, 32, 32, 3).astype("float32") / 255.0

# # Pasamos de dimensión 3 (n_samples, 28, 28) a dim 2 (n_samples, 748)
# x_train = x_train.reshape(len(x_train), -1)
# unlabeled_train = unlabeled_train.reshape(len(unlabeled_train), -1)
# x_test = x_test.reshape(len(x_test), -1)

# # convertimos de uint8 a float32
# x_train = x_train.astype("float32")
# unlabeled_train = unlabeled_train.astype("float32")
# x_test = x_test.astype("float32")

# # normalizamos y aplicamos PCA
# scaler = StandardScaler()

# x_train_scaled = scaler.fit_transform(x_train)
# unlabeled_train_scaled = scaler.transform(unlabeled_train)
# x_test_scaled = scaler.transform(x_test)

# pca = PCA(n_components=50, random_state=42)
# x_train_pca = pca.fit_transform(x_train_scaled)
# unlabeled_train_pca = pca.transform(unlabeled_train_scaled)
# x_test_pca = pca.transform(x_test_scaled)

#### 1. Entrena un modelo, creado sobre TensorFlow, haciendo uso únicamente de las instancias etiquetadas de entrenamiento. Dicho modelo debe de tener al menos cuatro capas densas y/o convolucionales.

In [ ]:
class MiCNN:

    def __init__(self):
        self.model = tf.keras.models.Sequential([
            tf.keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)),
            tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
            tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
            tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
            tf.keras.layers.Flatten(),
            tf.keras.layers.Dense(128, activation='relu'),
            tf.keras.layers.Dense(100, activation='softmax')
        ])

        optimizer = tf.keras.optimizers.Adam()

        self.model.compile(
            loss='sparse_categorical_crossentropy',
            optimizer=optimizer,
            metrics=['accuracy']
        )
    
    def fit(self, X, y, sample_weight=None):
        self.model.fit(X, y, 
                       epochs=10, 
                       batch_size=32,
                       sample_weight=sample_weight,
                       verbose=1)

    def predict(self, X):
        proba = self.model.predict(X)
        return proba.argmax(axis=1)
    
    def predict_proba(self, X):
        return self.model.predict(X)
    
    def score(self, X, y):
        loss, acc = self.model.evaluate(X, y, verbose=0)
        return acc

    def __del__(self):
        del self.model
        tf.keras.backend.clear_session()  # Necesario para liberar la memoria en GPU

In [ ]:
model = MiCNN()

model.fit(x_train, y_train)

acc = model.score(x_test, y_test)
print("Accuracy:", acc)

Epoch 1/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.0376 - loss: 4.3985
Epoch 2/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.1173 - loss: 3.8230
Epoch 3/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.1729 - loss: 3.4826
Epoch 4/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.2312 - loss: 3.1947
Epoch 5/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.2757 - loss: 2.9494
Epoch 6/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.3127 - loss: 2.7491
Epoch 7/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.3577 - loss: 2.5518
Epoch 8/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.3893 - loss: 2.3831
Epoch 9/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.4245 - loss: 2.2182
Epoch 10/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.4587 - loss: 2.0653
Accuracy: 0.2467000037431717


#### Responde las siguientes preguntas:
####    a. ¿Qué red has escogido? ¿Por qué? ¿Cómo la has entrenado?
####    b. ¿Cuál es el rendimiento del modelo en entrenamiento? ¿Y en prueba?
####    c. ¿Qué conclusiones sacas de los resultados detallados en el punto anterior?

#### 2. Entrena el mismo modelo, incorporando las instancias no etiquetadas de entrenamiento mediante la técnica de auto-aprendizaje. Opcionalmente, se ponderará cada instancia de entrada en función de su calidad (o certeza).

#### Responde las siguientes preguntas:
####    a. ¿Qué parámetros has definido para el entrenamiento?
####    b. ¿Cuál es el rendimiento del modelo en entrenamiento? ¿Y en prueba?
####    c. ¿Se mejoran los resultados obtenidos en el Ejercicio 1?
####    d. ¿Qué conclusiones sacas de los resultados detallados en los puntos anteriores?

#### 3. Entrena un modelo de aprendizaje semisupervisado de tipo autoencoder en dos pasos (primero el autoencoder, después el clasificador). La arquitectura del encoder debe ser exactamente la misma que la definida en los Ejercicios 1 y 2, a excepción del último bloque de capas.

#### Responde las siguientes preguntas:
####    a. ¿Cuál es la arquitectura del modelo? ¿Y sus hiperparámetros?
####    b. ¿Cuál es el rendimiento del modelo en entrenamiento? ¿Y en prueba?
####    c. ¿Se mejoran los resultados obtenidos en los Ejercicios 1 y 2?
####    d. ¿Qué conclusiones sacas de los resultados detallados en los puntos anteriores?

#### 4. Entrena un modelo de aprendizaje semisupervisado de tipo autoencoder en un paso (autoencoder y clasificador al mismo tiempo). La arquitectura del autoencoder será la misma que la definida en el Ejercicio 3, y la combinación de encoder y clasificador será igual a la arquitectura definida en el Ejercicio 1.

#### Responde las siguientes preguntas:
####    a. ¿Cuál es la arquitectura del modelo? ¿Y sus hiperparámetros?
####    b. ¿Cuál es el rendimiento del modelo en entrenamiento? ¿Y en prueba?
####    c. ¿Se mejoran los resultados obtenidos en los ejercicios anteriores?
####    d. ¿Qué conclusiones sacas de los resultados detallados en los puntos anteriores?

#### 5. Repite el mismo entrenamiento de los Ejercicios 2-4, pero eliminando las instancias no etiquetadas más atípicas con respecto a los datos etiquetados. Se cumplirán los siguientes puntos:

#### · La arquitectura de la red de clasificación en una clase será la misma a la utilizada en el clasificador del Ejercicio 1, a excepción de la capa de salida.

#### ·  Utiliza la técnica explicada en el Notebook 5, usando un valor de 𝑣 = 0,9.

#### Responde las siguientes preguntas:
####    a. ¿Se mejoran los resultados con respecto a los anteriores ejercicios? ¿Qué conclusiones sacas de estos resultados?

#### 6. Repite los Ejercicios 3-5 cambiando el autencoder por la técnica definida en el apartado “Hay vida más allá del autoencoder” del Notebook 4. Contesta a las preguntas de dichos ejercicios. Se cumplirán los siguientes puntos:

#### Responde las siguientes preguntas:
####    a. La arquitectura de la red será igual a la parte encoder del autencoder definido en los ejercicios anteriores
####    b. El modelo debe entrenar correctamente.